# Final Evaluation

In [1]:
from IPython.display import Markdown, display

from pathlib import Path
import duckdb

c2 = duckdb.connect()

## Step 1: Improve Your Workflow

### Dataset Scaling

Our original pipeline processed 10,000 products. See the code block below to verify the sample number we are using. (There was no need for us to increase or rebuild any of our indices.)

In [2]:
# verify at least 10,000 products

df = c2.execute("SELECT * FROM read_parquet('../data/processed/merged.parquet') WHERE rating_order = 1").df()
df.shape

(10000, 11)

### LLM Experiment

| Model | Name | Family | Size | Release Date |
|----------|----------|----------|----------|----------|
| 1 | Meta-Llama-3-8B-Instruct | Llama 3 | 8B parameters | April 18, 2024 |
| 2 | Llama-3.2-1B-Instruct | Llama 3.2 | 1B parameters | Sept. 25, 2024 |

We decided to stay within the same provider of models, Meta, in our case. Also, you'll see that one model family is a slightly new version of the other. The large difference between the two models is the number of parameters that were used to train the model. Model 1 is the one that we used originally for Milestone 2. It uses 8 times the number of paramters that model 2 does. We wanted to compare the responses between the two and see if this difference in parameters impacted subsequent results and if the "simpler" one would suffice.

You'll see the promt we used below - the same for both models to allow us to compare results fairly. It starts with a general system prompt and then adds context from our book sample and query from the user.

We have also included our outcome results in `results/evaluation_llms.md` which is rendered below for ease of viewing. Both of the models were working with the same top results.

Overall, we found that the Meta-Llama-3-8B-Instruct (model 1) performed better than the Llama-3.2-1B-Instruct (model 2). There were several cases that model 2 said there were no relevant books but there were ones that could tangentially cover the topic. This was seen in the "backpacking across europe" and "knitting crochet guide" queries. There was also a case where the response contradicted itself. For "books for kids learning geology," model 2's RAG Ensemble response said there were no books available but then provided a list of relevant books. We also noted that, for the "best way to start a business" query, model 2's RAG semantic response did not include any titles of books, just options on how to start a business. 

Despite model 2 not always returning book titles, or stating there are relevant results when there are, its responses were at least understandable. They weren't always reliable, but they weren't completely out-to-lunch talking about other topics or nonesense.

Ultimately, because of model 2's inconsistences and inability to give as-good-as-model-1 responses, we wouldn't use it in this context. Model 1, Meta-Llama-3-8B-Instruct, provides responses based on the top results that are relevant, without mistakeningly saying there are no books when, in fact, there are. This confirmed our suspicions that less parameters wasn't able to capture our context the best and led to worse outcomes.

#### Prompt used
```python
"""
You are a helpful Amazon shopping assistant.
You have access to a sample of books listed on Amazon.
Answer the question using only the following context:
Always provide the title and author, when available.
Do not include results that do not have both title and author.
If there isn't helpful information in the context, just say that
there are no relevant books available in the books sample.
Be very brief, answer is less then 40 words.

context:
{context_from_documents}

question:
{query_from_user}

Answer based on the Amazon datasets: 
"""
```

In [3]:
display(Markdown("evaluation_llms.md"))

================================================================================
## QUERY: backpacking across europe

### RAG Semantic

**RAG Response (Meta-Llama-3-8B-Instruct):** Ultrarunning Europe: Explorations of European Long Distance and Extreme Trails Through Running by Luke Tuttle. This guidebook features 38 curated trails for ultrarunning enthusiasts.

Iceland: Nature, Nurture & Adventure (Diary of a Traveling Black Woman: A Guide to International Travel) by Danielle Desir, also covers backpacking in Europe.

**RAG Response (Llama-3.2-1B-Instruct):** There are no relevant books available in the books sample for the question "backpacking across Europe".

#### Top Results (same for both)

1. Title: Ultrarunning Europe: Explorations of European Long Distance and Extreme Trails Through Running
2. Title: Iceland: Nature, Nurture & Adventure (Diary of a Traveling Black Woman: A Guide to International Travel)
3. Title: Camino de Santiago - Via Podiensis: Le Puy to the Pyrenees on the GR65
4. Title: Abgezockt und kaltgestellt
5. Title: Ireland: From the Air

### RAG Ensemble

**RAG Response (Meta-Llama-3-8B-Instruct)** Ultrarunning Europe: Explorations of European Long Distance and Extreme Trails Through Running, Luke Tuttle: This book showcases 38 trails across Europe, offering a diverse array of experiences for ultrarunners and trail enthusiasts.

Iceland: Nature, Nurture & Adventure (Diary of a Traveling Black Woman: A Guide to International Travel), Danielle Desir: This book offers a guide to exploring Iceland's landscapes, culture, and traditions for the traveling black woman.

Camino de...

**RAG Response (Llama-3.2-1B-Instruct):** There are no relevant books available in the books sample for the question "Backpacking across Europe".

#### Top Results (same for both)

1. Title: Ultrarunning Europe: Explorations of European Long Distance and Extreme Trails Through Running
2. Title: Abgezockt und kaltgestellt
3. Title: Iceland: Nature, Nurture & Adventure (Diary of a Traveling Black Woman: A Guide to International Travel)
4. Title: Camino de Santiago - Via Podiensis: Le Puy to the Pyrenees on the GR65
5. Title: Ireland: From the Air

================================================================================
## QUERY: books for kids learning geology

### RAG Semantic

**RAG Response (Meta-Llama-3-8B-Instruct)** How Do Wind and Water Change Earth? (Earth's Processes Close-up) by Natalie Hyde.
This book explains how the shape of Earth is changed by weathering and erosion.
It's suitable for young readers and includes a teacher's guide.

**RAG Response (Llama-3.2-1B-Instruct):** Here are the books you requested for kids learning geology:

1. "Geological Engineering" by Luis I. González de Vallejo
2. "The Secret Explorers and the Ice Age Adventure" by SJ King

#### Top Results (same for both)

1. Title: National Geographic Kids Sight Words 12-Book Reading Program
2. Title: Urbannature4kids Earth Science Lesson Plan: Earth Science for Elementary School-Aged Children in Grades K-4
3. Title: How Do Wind and Water Change Earth? (Earth's Processes Close-up)
4. Title: Geological Engineering
5. Title: The Secret Explorers and the Ice Age Adventure

### RAG Ensemble

**RAG Response (Meta-Llama-3-8B-Instruct)** How Do Wind and Water Change Earth? (Earth's Processes Close-up) by Natalie Hyde.

**RAG Response (Llama-3.2-1B-Instruct):** Unfortunately, there aren't any books available on Amazon that cover the topic of learning geology specifically for kids. However, here are a few additional books that may be of interest:

1. National Geographic Kids Geology Book
Author: Stephanie Snyder
Categories: Books, Nature & How It Works
Book details: This comprehensive book covers geology in a way that's easy for kids to understand, with colorful illustrations and simple explanations.
2. What's Inside a Volcano?
Author:  Claire

#### Top Results (same for both)

1. Title: National Geographic Kids Sight Words 12-Book Reading Program
2. Title: Urbannature4kids Earth Science Lesson Plan: Earth Science for Elementary School-Aged Children in Grades K-4
3. Title: How Do Wind and Water Change Earth? (Earth's Processes Close-up)
4. Title: Geological Engineering
5. Title: The Secret Explorers and the Ice Age Adventure

================================================================================
## QUERY: knitting crochet guide

### RAG Semantic

**RAG Response (Meta-Llama-3-8B-Instruct)** The Complete Guide to Modern Knitting and Crocheting by Alice Carroll.

**RAG Response (Llama-3.2-1B-Instruct):** - Complete Guide to Modern Knitting and Crocheting by Alice Carroll
- Needlelace: Designs and Techniques Classic and Contemporary by Anonymous
- Astounding Knits!: 101 Spectacular Knitted Creations and Daring Feats by Lela Nargi

#### Top Results (same for both)

1. Title: Complete Guide to Modern Knitting and Crocheting
2. Title: Needlelace: Designs and Techniques Classic and Contemporary
3. Title: 'Tis the Season to Crochet
4. Title: Astounding Knits!: 101 Spectacular Knitted Creations and Daring Feats
5. Title: Cakery Crochet - 9 Colorful Projects Using Yarn Cakes

### RAG Ensemble

**RAG Response (Meta-Llama-3-8B-Instruct)** Complete Guide to Modern Knitting and Crocheting by Alice Carroll. 
Cakery Crochet - 9 Colorful Projects Using Yarn Cakes by Leisure Arts. 
Astounding Knits!: 101 Spectacular Knitted Creations and Daring Feats by Lela Nargi.

**RAG Response (Llama-3.2-1B-Instruct):** I am not aware of any information that would suggest 'knitting crochet guide' is available on Amazon or in these sample books.

#### Top Results (same for both)

1. Title: Complete Guide to Modern Knitting and Crocheting
2. Title: Cakery Crochet - 9 Colorful Projects Using Yarn Cakes
3. Title: Needlelace: Designs and Techniques Classic and Contemporary
4. Title: 'Tis the Season to Crochet
5. Title: Astounding Knits!: 101 Spectacular Knitted Creations and Daring Feats

================================================================================
## QUERY: best way to start a business

### RAG Semantic

**RAG Response (Meta-Llama-3-8B-Instruct)** Plan B: The Real Deal Guide to Creating Your Business by Kathleen Rich-New. This book provides a mandatory reading guide to creating a business, helping you determine if you're ready and providing steps to choose the best option for your work lifestyle.

**RAG Response (Llama-3.2-1B-Instruct):** Based on the Amazon datasets, here are the options to start a business:

1. **Start a business**: This option involves creating a new business from scratch, which may require significant investment and effort.
2. **Buy a business**: This option involves purchasing an existing business, which can provide instant access to established operations and customer base.
3. **Buy a franchise**: This option involves buying a business that has already been established and operating, often with a proven business model and brand recognition.

These options...

#### Top Results (same for both)

1. Title: Plan B: The Real Deal Guide to Creating Your Business
2. Title: The Practical Guide to a Rapid Start-up (Launch Your Business in 20 Proven Steps)
3. Title: 10 Mistakes Startup Businesses Make & How You Can Avoid Them
4. Title: The Entrepreneurial Sell: Tools and Techniques Every Start-Up Needs to Succeed
5. Title: Selling Out to Your Level of Comfort

### RAG Ensemble

**RAG Response (Meta-Llama-3-8B-Instruct)** Plan B: The Real Deal Guide to Creating Your Business, Kathleen Rich-New. This book is a MUST-READ before starting a business, providing the necessary "business readiness" and due diligence for a successful start.

**RAG Response (Llama-3.2-1B-Instruct):** Based on the available books on Amazon, the best way to start a business is through a franchise, specifically one that aligns with your interests and skills. One of the top-rated books on this topic is "Shoestring Empire: Unlock the confidence and resources to start your own business!" by Ashley Elkins. This book provides a comprehensive guide to starting and growing a home-based baking business, as well as providing resources and motivation for turning a hobby into a profitable business.

#### Top Results (same for both)

1. Title: Plan B: The Real Deal Guide to Creating Your Business
2. Title: The Practical Guide to a Rapid Start-up (Launch Your Business in 20 Proven Steps)
3. Title: 10 Mistakes Startup Businesses Make & How You Can Avoid Them
4. Title: The Entrepreneurial Sell: Tools and Techniques Every Start-Up Needs to Succeed
5. Title: Selling Out to Your Level of Comfort

================================================================================
## QUERY: how to get out of debt

### RAG Semantic

**RAG Response (Meta-Llama-3-8B-Instruct)** Practically Independent: Practical Advice to Become Financially Independent, by Jay Rigler. The book outlines a strategy to escape living paycheck-to-paycheck, lower the cost of debt, and get paid for having money.

**RAG Response (Llama-3.2-1B-Instruct):** There are no relevant books available in the provided sample dataset.

#### Top Results (same for both)

1. Title: Stop Overspending: Why Most People Can't Save Money
2. Title: Practically Independent: Practical Advice to Become Financially Independent
3. Title: Practically Independent: Practical Advice to Become Financially Independent
4. Title: Grab Your Slice of Financial Independence
5. Title: How To Retire Healthy: Thoughts On Personal Finance

### RAG Ensemble

**RAG Response (Meta-Llama-3-8B-Instruct)** "How to get out of debt?
Consider:
- Practically Independent: Practical Advice to Become Financially Independent (Jay Rigler) 
- Stop Overspending: Why Most People Can't Save Money 
- Living Abundantly on Limited Assets (Anton Tkachenko)"

**RAG Response (Llama-3.2-1B-Instruct):** Based on the provided context, here are the answers to the question "how to get out of debt" for each book:

1. "Practically Independent: Practical Advice to Become Financially Independent" by Jay Rigler
   - To get out of debt, start by creating a budget, tracking expenses, and identifying areas where cuts can be made.
   - Consider paying off high-interest debt first, and use the snowball method or avalanche method to pay off debts.
   - Build an...

#### Top Results (same for both)

1. Title: Practically Independent: Practical Advice to Become Financially Independent
2. Title: Stop Overspending: Why Most People Can't Save Money
3. Title: Grab Your Slice of Financial Independence
4. Title: How To Retire Healthy: Thoughts On Personal Finance
5. Title: THE WAR ON APHASIA: SAVING YOUR HOME FROM FORECLOSURE WAY OF THE CREDITOR

## Step 2: Additional Feature

### What We Implemented

We chose "Option 4: Deploy your Application." The service that we decided to go with was Posit Connect Cloud. This was taught to us in an earlier block and is fairly straightforward to implement. It works well with both our different search systems. We were also able to add a "Secret variable" that connects to HuggingFace, allowing our RAG pipeline to function as expected.

Link to dashboard: [https://019dac27-bcb0-d837-c305-f05496dcbe18.share.connect.posit.cloud](https://019dac27-bcb0-d837-c305-f05496dcbe18.share.connect.posit.cloud)

## Step 3: Improve Documentation and Code Quality

### Documentation Update
- Summary of `README` improvements

### Code Quality Changes
- Summary of cleanups

## Step 4: Cloud Deployment Plan

In this cloud deployment plan will assume we are using AWS to deploy our dashboard. This will be a very lightweight deployment. We will be expecting just a handful of concurrent users. For this initial AWS deployment we will attempt to be as low cost as possible.

### 1. Data storage

We will locally store the raw and processed data. These are not needed for the app to run. In the future if we need more processing power to update the vector store and BM25 index we may move the raw and processed data to S3.

The vector store and BM25 will be saved in S3. This will make them easily accessible to our EC2 instance which will run the app.

### 2. Compute

Initially the app with run on a single EC2 instance. We expect this to be able to handle a small amount of concurrent uses. This will be tested before making the app public. We will use the smallest/cheapest EC2 instance that can run the app with a handful of concurrent users. If one EC2 instance can not handle this then we will use an AWS Elastic Beanstalk to deploy the app while managing capacity. 

We will try using uvicorn to handle multiple user. If this is unable to handle to workload required by a shiny app we will attempt to use Shiny Server.

LLM inference will continue to be handled via API calls to hugging face. The api key will managed with AWS Systems Manager Parameter Store. 

### 3. Streaming/Updates

Initially new products will be added by manually downloading the current raw data. Processing the data and creating the BM25 index and vector store will be done locally. The updated BM25 index and vector store will then be manually uploaded to S3 so the app can have access to the most up to date data.

If the app continues to draw users we will eventually switch to an automated process. Will will use Amazon event bridge to trigger the update on some schedule (daily or weekly) and AWS lambda to run the python script to update the data and save new BM25 index and vector store to S3.